# 09 — Classification Results

Produces **paper-quality figures** from the evaluation tables written by NB08.
No model loading, no inference — reads saved CSVs from Drive.

**Figures produced** (saved to `outputs/figures/classification/<dataset>/<exp>/`):

| Figure | Description |
|---|---|
| `confusion_pair.png` | Eval A + Eval B confusion matrices side-by-side |
| `per_class_f1.png` | Grouped F1 bars per class + macro, with fold std error bars |
| `eval_gap.png` | Per-fold Eval A vs Eval B gap chart |
| `sample_patches.png` | 3×N grid of correctly classified tumor patches |

**Prerequisites:**
- NB08 Eval A must have run: `outputs/tables/classification/<dataset>/<exp>/eval_gt/cv_results.csv`
- NB08 Eval B is optional (gap chart + Eval B confusion matrix are skipped if missing)

**Runtime:** CPU-only, no GPU needed.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists('/content/senior_project'):
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    url = 'https://github.com/salemaker47/senior_project.git'
    if token:
        url = url.replace('https://', f'https://{token}@', 1)
    os.system(f'git clone {url} /content/senior_project')
if '/content/senior_project' not in sys.path:
    sys.path.insert(0, '/content/senior_project')

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url='https://github.com/salemaker47/senior_project.git',
)
print(f'DRIVE_ROOT: {DRIVE_ROOT}')
print(f'REPO_ROOT:  {REPO_ROOT}')

## Cell 3 — Config

Set `RECIPE`, `DATASET`, `SPLIT_SCHEME`, and `SEG_EXPERIMENT` to match the NB08 run
whose results you want to visualise.

**Knobs:**
- `N_PATCH_EXAMPLES` — columns in the sample-patch grid (default 3 = 9 patches total)
- `PATCH_KIND` — `"correct"` | `"incorrect"` | `"random"`

In [ ]:
from configs.cls.reference_experiments import get_experiment, REFERENCE_RECIPES

# ----- Match NB07/NB08 settings -----
RECIPE        = 'cls01_resnet50'
DATASET       = 'figshare'
SPLIT_SCHEME  = 'image_level'
SEG_EXPERIMENT = '07_unetpp_effb4_dicebce_image_level'  # for Eval B

# ----- Figure knobs -----
N_PATCH_EXAMPLES = 3      # columns per class in the sample-patch grid
PATCH_KIND       = 'correct'   # 'correct' | 'incorrect' | 'random'

# ---------------------------------------------------------------------------
EXPERIMENT = get_experiment(
    RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1,
)
EXP_NAME = EXPERIMENT['name']

print(f'Experiment:     {EXP_NAME}')
print(f'Dataset:        {DATASET}')
print(f'Seg experiment: {SEG_EXPERIMENT}')
print(f'Patch grid:     {N_PATCH_EXAMPLES} per class  ({PATCH_KIND})')
print(f'Available recipes: {REFERENCE_RECIPES}')

## Cell 4 — Load evaluation tables from Drive

CSVs are read directly from Drive (they are small).  
`copy_to_local` is called only to provide images for the patch grid in Cell 9.

In [ ]:
import pandas as pd
from pathlib import Path
from src.notebook_setup import copy_to_local
from src.file_utils     import cls_eval_paths, experiment_root_paths

# ---- Eval A paths (required) ----
paths_a = cls_eval_paths(DRIVE_ROOT, DATASET, EXP_NAME, mask_source='gt')
tables_dir_a = paths_a['tables_dir']

# ---- Eval B paths (optional) ----
paths_b = cls_eval_paths(
    DRIVE_ROOT, DATASET, EXP_NAME,
    mask_source='predicted', seg_experiment_name=SEG_EXPERIMENT,
)
tables_dir_b = paths_b['tables_dir']

def _load_csv(path: Path, label: str):
    if path.exists():
        return pd.read_csv(path)
    print(f'  WARNING: {label} not found at {path}')
    return None

# Eval A
cv_results_a   = _load_csv(tables_dir_a / 'cv_results.csv',   'Eval A cv_results')
cv_summary_a   = _load_csv(tables_dir_a / 'cv_summary.csv',   'Eval A cv_summary')
cv_confusion_a = _load_csv(tables_dir_a / 'cv_confusion.csv', 'Eval A cv_confusion')
cv_per_image_a = _load_csv(tables_dir_a / 'cv_per_image.csv', 'Eval A cv_per_image')

if cv_confusion_a is not None:
    # 'true_class' is the named index column written by NB08 via index_label='true_class'
    cv_confusion_a = cv_confusion_a.set_index('true_class')

# Eval B (optional)
cv_results_b   = _load_csv(tables_dir_b / 'cv_results.csv',   'Eval B cv_results')
cv_summary_b   = _load_csv(tables_dir_b / 'cv_summary.csv',   'Eval B cv_summary')
cv_confusion_b = _load_csv(tables_dir_b / 'cv_confusion.csv', 'Eval B cv_confusion')
cv_per_image_b = _load_csv(tables_dir_b / 'cv_per_image.csv', 'Eval B cv_per_image')

if cv_confusion_b is not None:
    cv_confusion_b = cv_confusion_b.set_index('true_class')

has_b = cv_results_b is not None

print(f'Eval A tables:  {"OK" if cv_results_a is not None else "MISSING"}')
print(f'Eval B tables:  {"OK" if cv_results_b is not None else "MISSING (gap chart skipped)"}')

# ---- Local SSD for images (patch grid) ----
LOCAL_ROOT = copy_to_local(DRIVE_ROOT, datasets=[DATASET])
print(f'LOCAL_ROOT: {LOCAL_ROOT}')

# ---- Output figure dir (experiment-level) ----
exp_root_paths = experiment_root_paths(
    LOCAL_ROOT, task=EXPERIMENT['task'],
    dataset=DATASET, experiment_name=EXP_NAME,
)
FIG_DIR = exp_root_paths['figures']
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Figure output dir: {FIG_DIR}')

## Cell 5 — Summary table

In [ ]:
from IPython.display import display
import warnings

METRIC_COLS = [
    'macro_f1_mean', 'macro_f1_std',
    'accuracy_mean', 'accuracy_std',
    'f1_meningioma_mean', 'f1_glioma_mean', 'f1_pituitary_mean',
]
FMT = {c: '{:.4f}' for c in METRIC_COLS}

rows = []
for label, df in [('Eval A (GT)', cv_summary_a), ('Eval B (pred)', cv_summary_b)]:
    if df is None:
        continue
    row = {'variant': label}
    for c in METRIC_COLS:
        row[c] = df[c].iloc[0] if c in df.columns else float('nan')
    rows.append(row)

if rows:
    summary_display = pd.DataFrame(rows).set_index('variant')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        display(summary_display.style.format(FMT, na_rep='—'))
else:
    print('No summary tables loaded — run NB08 first.')

## Cell 6 — Confusion matrix pair

In [ ]:
import matplotlib.pyplot as plt
from src.cls_vis_utils import plot_confusion_matrix, plot_confusion_pair

if cv_confusion_a is None:
    print('Eval A confusion matrix not loaded — skipping.')
elif has_b and cv_confusion_b is not None:
    # Side-by-side pair
    fig = plot_confusion_pair(
        cv_confusion_a,
        cv_confusion_b,
        title_a='Eval A — GT masks',
        title_b=f'Eval B — {SEG_EXPERIMENT}',
        suptitle=f'{EXP_NAME}  ·  {DATASET}  — confusion matrices (sum across folds)',
        save_path=FIG_DIR / 'confusion_pair.png',
        show=True,
    )
    print(f'saved {FIG_DIR / "confusion_pair.png"}')
else:
    # Single matrix (Eval A only)
    fig = plot_confusion_matrix(
        cv_confusion_a,
        title=f'{EXP_NAME} — Eval A (GT masks)',
        save_path=FIG_DIR / 'confusion_eval_a.png',
        show=True,
    )
    print(f'saved {FIG_DIR / "confusion_eval_a.png"}')

## Cell 7 — Per-class F1 bar chart

In [ ]:
from src.cls_vis_utils import plot_per_class_f1

if cv_results_a is None:
    print('Eval A cv_results not loaded — skipping.')
else:
    n_folds = cv_results_a['fold'].nunique()
    fig = plot_per_class_f1(
        cv_results_a,
        cv_results_b=cv_results_b if has_b else None,
        title=f'{EXP_NAME}  ·  {DATASET}  — per-class F1  ({n_folds}-fold CV)',
        save_path=FIG_DIR / 'per_class_f1.png',
        show=True,
    )
    print(f'saved {FIG_DIR / "per_class_f1.png"}')

## Cell 8 — Eval A vs Eval B gap chart

In [ ]:
from src.cls_vis_utils import plot_eval_gap

if not has_b or cv_results_b is None:
    print('Eval B cv_results not available — skipping gap chart.')
    print('Run NB08 Eval B section first.')
else:
    fig = plot_eval_gap(
        cv_results_a,
        cv_results_b,
        title=f'{EXP_NAME}  ·  {DATASET}  — Eval A vs Eval B macro F1 gap',
        seg_experiment_name=SEG_EXPERIMENT,
        save_path=FIG_DIR / 'eval_gap.png',
        show=True,
    )
    print(f'saved {FIG_DIR / "eval_gap.png"}')

## Cell 9 — Sample patch grid

Shows `N_PATCH_EXAMPLES` patches per class from the Eval A test set.
Green border = correctly classified; red border = misclassified.
Change `PATCH_KIND` in Cell 3 to `"incorrect"` to visualise failure cases.

In [ ]:
from src.cls_vis_utils import plot_sample_patches

if cv_per_image_a is None:
    print('Eval A cv_per_image not loaded — skipping patch grid.')
else:
    fig = plot_sample_patches(
        cv_per_image=cv_per_image_a,
        project_root=LOCAL_ROOT,
        mask_source='gt',
        predictions_dir=None,
        n_per_class=N_PATCH_EXAMPLES,
        kind=PATCH_KIND,
        target_size=EXPERIMENT['patch_size'],
        padding_frac=EXPERIMENT['padding_frac'],
        random_state=42,
        title=(
            f'{EXP_NAME}  ·  {DATASET}\n'
            f'{PATCH_KIND} predictions  |  GT masks  |  green = correct, red = incorrect'
        ),
        save_path=FIG_DIR / 'sample_patches.png',
        show=True,
    )
    print(f'saved {FIG_DIR / "sample_patches.png"}')

## Cell 10 — Sync figures to Drive

In [ ]:
from src.notebook_setup import sync_outputs_to_drive

sync_outputs_to_drive(
    drive_root=DRIVE_ROOT, local_root=LOCAL_ROOT,
    task=EXPERIMENT['task'], dataset=DATASET,
    experiment_name=EXP_NAME,
    categories=['figures'],
)
print('sync complete')

In [ ]:
SYNC_OK = True   # set manually after confirming the sync cell above completed
if SYNC_OK:
    from google.colab import runtime
    print('Disconnecting runtime in 3 seconds...')
    import time; time.sleep(3)
    runtime.unassign()
else:
    print('SYNC_OK is False — staying connected.')